In [1]:
# @title Setup and Imports
import pandas as pd
import evaluate
import re
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import matplotlib.pyplot as plt
import torch
import re
from neo4j import GraphDatabase
from collections import defaultdict

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm


from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
try:
    from google.colab import userdata
    get_secret = userdata.get
except ImportError:
    # Fallback: define a dummy get_secret for local use
    def get_secret(key):
        return None

def get_env(key: str, default: str = None):
    """Tries os.getenv first, then Colab userdata if available."""
    return os.getenv(key) or get_secret(key) or default


DB_HOST = get_env("POSTGRES_DB_HOST")
DB_PORT = int(get_env("POSTGRES_DB_PORT"))
DB_NAME = get_env("POSTGRES_DB_NAME")
DB_USER = get_env("POSTGRES_DB_USER")
DB_PASS = get_env("POSTGRES_DB_PASS")

NEO4J_URI  = get_env("NEO4J_URI")   # bolt:// or neo4j://
NEO4J_USER = get_env("NEO4J_USER")
NEO4J_PASS = get_env("NEO4J_PASS")
NEO4J_DATABASE = get_env("NEO4J_DATABASE")

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"





In [2]:
from __future__ import print_function
import os.path
import pickle

from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

# -------------------------------------------------
# Scopes: Docs + Drive (file-level)
# -------------------------------------------------
SCOPES = [
    "https://www.googleapis.com/auth/documents",
    "https://www.googleapis.com/auth/drive.file",
]

def get_user_creds():
    creds = None
    # token.pickle stores the user’s access/refresh token
    if os.path.exists("token.pickle"):
        with open("token.pickle", "rb") as token:
            creds = pickle.load(token)
    # If no valid creds, run the OAuth flow
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                "oauth_client_secret.json",  # downloaded from Cloud Console
                SCOPES,
            )
            creds = flow.run_local_server(port=0)
        # Save the credentials for next run
        with open("token.pickle", "wb") as token:
            pickle.dump(creds, token)
    return creds





In [3]:
source_sql = """
SELECT
 policies.policy_id,
 policies.policy_title, policies.section_id, policies.section_title, source_framework,
org_text,
V1_5_OUTPUT,
V2_5_OUTPUT,
V3_5_OUTPUT,
V3_7_OUTPUT,
V4_0_OUTPUT,
split_label
FROM (
SELECT
policy_id, section_id, section_title,
MAX(case when version = 'v1.5' then output else null end) As V1_5_OUTPUT,
MAX(case when version = 'v2.5' then output else null end) As V2_5_OUTPUT,
MAX(case when version = 'v3.5' then output else null end) As V3_5_OUTPUT,
MAX(case when version = 'v3.7' then output else null end) As V3_7_OUTPUT,
MAX(case when version = 'v4.0' then output else null end) As V4_0_OUTPUT
FROM (
SELECT
policy_id, section_id, section_title, user_prompt, version, system_prompt, output, time_taken, token_count
FROM prompt_responses
UNION ALL
SELECT
policy_id, section_id, section_title, user_prompt, version, system_prompt, output, time_taken, token_count
FROM prompt_responses2
) as D
group by policy_id, section_id, section_title
) AS DD
JOIN (
SELECT DISTINCT policy_title, policy_id, section_id, section_title, source_framework
, string_agg(clause_text, '\n' ORDER BY ROW_NUMBER) AS org_text  FROM policy_lines
GROUP BY policy_id, section_id, section_title, source_framework, policy_title
) as policies on DD.policy_id = policies.policy_id and DD.section_id = policies.section_id
JOIN policy_splits on policies.policy_id = policy_splits.policy_id

WHERE policy_title IN ('Enterprise IT Risk Management Policy', 'Server Security Policy') and policies.section_title in ('PURPOSE', 'Roles and Responsibilities')

"""

In [4]:
full_df = pd.read_sql(text(source_sql), connection_url)

In [5]:
# %pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib pandas


In [6]:
import numpy as np
import pandas as pd

def safe(x, eps=1e-6):
    """Clamp to at least eps; works for scalars, Series, ndarrays."""
    return np.maximum(x, eps)

def add_pillars_and_mds(scores_df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds:
      - S_coverage
      - S_semantic_faith
      - S_surface
      - S_MDS  (weighted harmonic mean of the three pillars)

    Returns a new DataFrame with these columns.
    """
    df = scores_df.copy()

    # --- 1) Coverage pillar ---
    df["S_coverage"] = (
        0.25 * df["S_keyword"] +
        0.30 * df["S_ontology_direct"] +
        0.30 * df["S_ontology_inferred"] +
        0.15 * df["S_ontology"]
    )

    # --- 2) Semantic + faithfulness pillar ---
    df["S_semantic_faith"] = (
        0.40 * df["S_semantic_bge"] +
        0.40 * df["S_ragas_faithfulness"] +
        0.20 * df["S_ragas_relevancy"]
    )

    # --- 3) Surface pillar ---
    if "S_spacy" in df.columns:
        df["S_surface"] = (
            0.40 * df["S_lexical"] +
            0.30 * df["S_length"] +
            0.30 * df["S_spacy"]
        )
    else:
        df["S_surface"] = 0.5 * df["S_lexical"] + 0.5 * df["S_length"]

    # --- 4) MDS: weighted harmonic mean of the three pillars ---
    w_cov, w_sem, w_surf = 0.35, 0.45, 0.20

    denom = (
        w_cov  / safe(df["S_coverage"]) +
        w_sem  / safe(df["S_semantic_faith"]) +
        w_surf / safe(df["S_surface"])
    )

    df["S_MDS"] = 1.0 / denom

    return df


In [16]:
from googleapiclient.discovery import build
from google.oauth2 import service_account
import pandas as pd



# -------------------------------------------------
# Helper: get the last table in the doc
# -------------------------------------------------
def get_last_table(doc_id):
    doc = docs_service.documents().get(documentId=doc_id).execute()
    tables = []
    for el in doc.get("body", {}).get("content", []):
        if "table" in el:
            tables.append(el["table"])
    return tables[-1] if tables else None


def get_scores(section_id:str, model_name:str):
    # Placeholder function to return dummy scores
    # Replace with actual logic to compute or fetch scores
    sql = f"""
    select 
"S_keyword", "S_ontology", "S_ontology_direct", "S_ontology_inferred", "S_ragas_faithfulness", "S_ragas_relevancy", "S_ragas"
, "S_spacy", "S_semantic_semantic", "MDS_semantic"
, "{"S_semantic_bge" if model_name == 'v1.5' else 'S_sem" as "S_semantic_bge'}"
, "{"S_lexical" if model_name == 'v1.5' else 'S_lex" as "S_lexical'}"
, "{"S_length" if model_name == 'v1.5' else 'S_len" as "S_length'}"

from {'scores2' if model_name == 'v1.5' else 'scores4' if model_name == 'v2.5' else 'scores6'}
    where section_id = '{section_id}'
    and model_name = '{model_name}'
    """
    scores_df = pd.read_sql(
        text(sql),
        connection_url,
        params={"section_id": section_id, "model_name": model_name},
    )

    if scores_df.empty:
        return {"SCORES": "N/A"}

    # Compute pillars + S_MDS
    scores_df = add_pillars_and_mds(scores_df)
    row = scores_df.iloc[0]

    # Order the output a bit for SME readability
    metric_order = [
        # base metrics
        "S_lexical",
        "S_length",
        "S_keyword",
        "S_ontology",
        "S_ontology_direct",
        "S_ontology_inferred",
        "S_ragas_faithfulness",
        "S_ragas_relevancy",
        "S_ragas",
        "S_spacy",
        "S_semantic_bge",
        # pillars
        "S_coverage",
        "S_semantic_faith",
        "S_surface",
        # composite
        "S_MDS",
    ]

    results = {}
    for key in metric_order:
        if key not in row.index:
            continue
        val = row[key]
        if pd.isna(val):
            continue
        results[key] = val
    if not results:
        return {"SCORES": "N/A"}
    return results

    


def build_text_requests_for_table(doc_id, table_grid, font_size_pt=10):
    """
    table_grid: list of rows; each row is [cell_text_col1, cell_text_col2, ...]
    For each non-empty cell, we:
      1) insert text
      2) immediately apply font size to exactly that text.
    """
    # Re-fetch doc to see the last table we just inserted
    doc = docs_service.documents().get(documentId=doc_id).execute()
    body = doc.get("body", {}).get("content", [])

    last_table = None
    for el in body:
        if "table" in el:
            last_table = el["table"]
    if last_table is None:
        raise RuntimeError("No table found in document.")

    table_rows = last_table["tableRows"]

    # Collect (index, text) info first so we can sort by index DESC
    pending = []

    for r_idx, row_vals in enumerate(table_grid):
        row = table_rows[r_idx]
        for c_idx, cell_text in enumerate(row_vals):
            if cell_text is None:
                continue
            text = str(cell_text)
            if text.strip() == "":
                continue

            cell = row["tableCells"][c_idx]
            cell_content = cell.get("content", [])
            if not cell_content:
                continue

            para = cell_content[0].get("paragraph", {})
            elements = para.get("elements", [])
            if not elements:
                continue

            elem = elements[-1]
            start = elem.get("startIndex", 0)
            end = elem.get("endIndex", start + 1)

            # Compute a safe insert index inside this paragraph
            length = max(1, end - start)
            if length <= 1:
                insert_index = end - 1
            else:
                insert_index = min(start + 1, end - 1)

            pending.append({
                "index": insert_index,
                "text": text,
            })

    # VERY IMPORTANT: sort by index DESC so earlier inserts don't shift later ones
    pending.sort(key=lambda x: x["index"], reverse=True)

    # Build the batchUpdate requests
    requests = []
    for item in pending:
        idx = item["index"]
        text = item["text"]
        text_len = len(text)

        # 1) insert the text
        requests.append({
            "insertText": {
                "location": {"index": idx},
                "text": text,
            }
        })

        # 2) apply font size to exactly that span
        requests.append({
            "updateTextStyle": {
                "range": {
                    "startIndex": idx,
                    "endIndex": idx + text_len,
                },
                "textStyle": {
                    "fontSize": {
                        "magnitude": float(font_size_pt),
                        "unit": "PT",
                    },
                },
                "fields": "fontSize",
            }
        })

    return requests



# -------------------------------------------------
# 3. Create the Google Doc
# -------------------------------------------------





In [8]:
def get_last_table_element(doc_id):
    """Return the last table element (including startIndex) from the doc body."""
    doc = docs_service.documents().get(documentId=doc_id).execute()
    last_table_el = None
    for el in doc.get("body", {}).get("content", []):
        if "table" in el:
            last_table_el = el
    return last_table_el


In [9]:
def set_column_widths(doc_id, table_start_index, col_widths_pt):
    """
    col_widths_pt = { columnIndex: width_in_points }
    Example: {0: 120, 1: 320, 2: 100}
    """
    requests = []

    for col_idx, width in col_widths_pt.items():
        requests.append({
            "updateTableColumnProperties": {
                "tableStartLocation": {"index": table_start_index},
                "columnIndices": [col_idx],
                "tableColumnProperties": {          # 🔑 this was wrong before
                    "widthType": "FIXED_WIDTH",
                    "width": {
                        "magnitude": float(width),
                        "unit": "PT",
                    },
                },
                "fields": "width,widthType"        # include both
            }
        })

    if requests:
        docs_service.documents().batchUpdate(
            documentId=doc_id,
            body={"requests": requests},
        ).execute()


In [10]:
import time
from googleapiclient.errors import HttpError

def safe_batch_update(docs_service, doc_id, requests, max_retries=10, base_sleep=1.0):
    """
    Wrapper around documents().batchUpdate with retry & exponential backoff
    when we hit 429 (RATE_LIMIT_EXCEEDED).
    """
    for attempt in range(max_retries):
        try:
            return docs_service.documents().batchUpdate(
                documentId=doc_id,
                body={"requests": requests},
            ).execute()
        except HttpError as e:
            status = getattr(e.resp, "status", None)
            if status == 429:
                # Exponential backoff
                wait = base_sleep * (2 ** attempt)
                print(f"Hit Docs write quota (429). Sleeping {wait:.1f}s then retrying...")
                time.sleep(wait)
                continue
            # Any other error: re-raise
            raise

    raise RuntimeError("Exceeded max retries for Docs batchUpdate")


In [11]:
# df[df['split_label'] == 'test']

In [12]:
df = full_df.sample(frac=1, random_state=42).reset_index(drop=True)

df = full_df.rename(columns={
    "policy_title": "policy_name",
    "section_title": "section_title",
    "source_framework": "framework_name",
    "org_text": "target_response",
    "v1_5_output": "v1_5_text",
    "v2_5_output": "v2_5_text",
    "v3_5_output": "v3_5_text",
    "v3_7_output": "v3_7_text",
        
    
})
df=df.fillna("")
df = df[df['split_label'] == 'test']


In [13]:
creds = get_user_creds()
docs_service = build("docs", "v1", credentials=creds)

# -----------------------
# 3. Create the document
# -----------------------
doc = docs_service.documents().create(body={"title": "DataFrame Export"}).execute()
doc_id = doc["documentId"]

print(f"Done. https://docs.google.com/document/d/{doc_id}")

Done. https://docs.google.com/document/d/1sB8_WZawucqZfch-ML5oE6CLpMQB-97bpAqvY0xWIRM


In [14]:
# get_scores("CL_0001-PMP-v1.0:S5", "v2.5")

In [17]:
# -------------------------------------------------
# 4. For each DataFrame row: build table + fill + page break
# -------------------------------------------------

# Only use test split
df_test = df[df["split_label"] == "test"].reset_index(drop=True)

num_rows = len(df_test)

for i, (_, row) in enumerate(df_test.iterrows()):

    # 4a. Build the table grid for this policy
    header_text = f"\"{row['policy_name']}\": \"{row['section_title']} ({row['framework_name']})\""
    v1_5_scores = get_scores(row["section_id"], "v1.5")
    v2_5_scores = get_scores(row["section_id"], "v2.5")
    v3_7_scores = get_scores(row["section_id"], "v3.7")

    metric_order = [
        "S_coverage",
        "S_semantic_faith",
        "S_surface",
        "S_MDS",
        "S_ragas_faithfulness",
        "S_ragas_relevancy",
    ]
     # ---------- V1.5 formatting ----------
    v1_5_lines = []
    for key in metric_order:
        if key not in v1_5_scores:
            continue
        val = v1_5_scores[key]
        if pd.isna(val):
            continue
        tabs = 2 if len(key) <= 13 else 1
        tabs_str = "\t" * tabs
        if key == "S_ragas_faithfulness":
            v1_5_lines.append('-----------------------------------------')
        v1_5_lines.append(f"{key}:{tabs_str}{val:.4f}")

    v1_5_display = "\n".join(v1_5_lines)

    # ---------- V2.5 formatting ----------
    v2_5_lines = []
    for key in metric_order:
        if key not in v2_5_scores:
            continue
        val = v2_5_scores[key]
        if pd.isna(val):
            continue
        base = v1_5_scores.get(key, None)
        if base is not None and not pd.isna(base):
            delta = val - base
            delta_str = f" (Δ {delta:+.4f})\n"   # e.g. " (+0.1195)"
        else:
            delta_str = ""

        tabs = 2 if len(key) <= 13 else 1
        tabs_str = "\t" * tabs
        if key == "S_ragas_faithfulness":
            v2_5_lines.append('-----------------------------------------')
        v2_5_lines.append(f"{key}:{tabs_str}{val:.4f}{delta_str}")

    v2_5_display = "\n".join(v2_5_lines)

    # ---------- V3.7 formatting with delta vs V2.5 ----------
    v3_7_lines = []
    for key in metric_order:
        if key not in v3_7_scores:
            continue
        val3 = v3_7_scores[key]
        if pd.isna(val3):
            continue

        base = v1_5_scores.get(key, None)
        base_v2 = v2_5_scores.get(key, None)
        delta_str = ""

        if base is not None and not pd.isna(base):
            delta = val3 - base
            delta_str = f" (Δv1 {delta:+.4f})\n"   # e.g. " (+0.1195)"
        if base_v2 is not None and not pd.isna(base_v2):
            delta = val3 - base_v2
            delta_str += f"(Δv2 {delta:+.4f})\n"   # e.g. " (+0.1195)"
        

        tabs = 2 if len(key) <= 13 else 1
        tabs_str = "\t" * tabs
        if key == "S_ragas_faithfulness":
            v3_7_lines.append('-----------------------------------------')
        v3_7_lines.append(f"{key}:{tabs_str}{val3:.4f}{delta_str}")

    v3_7_display = "\n".join(v3_7_lines)


    

    table_grid = [
        # Row 0: header row (merged across all 3 columns)
        [header_text, "", ""],

        # Row 1: target response (merged across all 3 columns)
        [f"Target Response:\n{row['target_response']}", "", ""],

        # Row 2: V1.5
        ["V1.5", row["v1_5_text"], v1_5_display],

        # Row 3: V2.5
        
        ["V2.5", row["v2_5_text"], v2_5_display],

        # Row 4: V3.5
        ["V3.7", row["v3_7_text"], v3_7_display],
    ]

    # 4b. First write: insert the empty table (5 rows × 3 columns)
    insert_table_req = [{
        "insertTable": {
            "rows": len(table_grid),
            "columns": 3,
            "endOfSegmentLocation": {}  # append at end of doc
        }
    }]

    safe_batch_update(docs_service, doc_id, insert_table_req)

    # 4c. Get the last table's start index
    last_table_el = get_last_table_element(doc_id)
    if last_table_el is None:
        raise RuntimeError("No table element found after insertTable.")
    table_start_index = last_table_el["startIndex"]

    # 4d. Second write: layout operations (merges + column widths)
    layout_ops = []

    # Merge row 0 across all 3 columns
    layout_ops.append({
        "mergeTableCells": {
            "tableRange": {
                "tableCellLocation": {
                    "tableStartLocation": {"index": table_start_index},
                    "rowIndex": 0,
                    "columnIndex": 0,
                },
                "rowSpan": 1,
                "columnSpan": 3,
            }
        }
    })

    # Merge row 1 across all 3 columns
    layout_ops.append({
        "mergeTableCells": {
            "tableRange": {
                "tableCellLocation": {
                    "tableStartLocation": {"index": table_start_index},
                    "rowIndex": 1,
                    "columnIndex": 0,
                },
                "rowSpan": 1,
                "columnSpan": 3,
            }
        }
    })

    # Column widths: narrow label col, wide text col, spare col
    layout_ops.extend([
        {
            "updateTableColumnProperties": {
                "tableStartLocation": {"index": table_start_index},
                "columnIndices": [0],
                "tableColumnProperties": {
                    "widthType": "FIXED_WIDTH",
                    "width": {"magnitude": 35.0, "unit": "PT"},
                },
                "fields": "width,widthType",
            }
        },
        {
            "updateTableColumnProperties": {
                "tableStartLocation": {"index": table_start_index},
                "columnIndices": [1],
                "tableColumnProperties": {
                    "widthType": "FIXED_WIDTH",
                    "width": {"magnitude": 300.0, "unit": "PT"},
                },
                "fields": "width,widthType",
            }
        },
        {
            "updateTableColumnProperties": {
                "tableStartLocation": {"index": table_start_index},
                "columnIndices": [2],
                "tableColumnProperties": {
                    "widthType": "FIXED_WIDTH",
                    "width": {"magnitude": 150.0, "unit": "PT"},
                },
                "fields": "width,widthType",
            }
        },
    ])

    safe_batch_update(docs_service, doc_id, layout_ops)

    # 4e. Third write: text insert + font-size styling (+ optional page break)
    text_requests = build_text_requests_for_table(
        doc_id,
        table_grid,
        font_size_pt=10
    )

    # Add a page break after this table except for the last one
    if i < num_rows - 1:
        text_requests.append({
            "insertPageBreak": {
                "endOfSegmentLocation": {}
            }
        })

    safe_batch_update(docs_service, doc_id, text_requests)

    # (Optional) small sleep to stay well below per-minute write quota
    # time.sleep(0.2)

print(f"Done. https://docs.google.com/document/d/{doc_id}")


Done. https://docs.google.com/document/d/1sB8_WZawucqZfch-ML5oE6CLpMQB-97bpAqvY0xWIRM
